In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import Spectral_Preprocessing as sp

from argparse import ArgumentParser, Namespace
from scipy import stats

In [2]:
parser = ArgumentParser()
parser.add_argument("--range-min-ir", dest= "range_min_ir", type = int, default = 400)
parser.add_argument("--range-max-ir", dest= "range_max_ir", type = int, default = 4000)
parser.add_argument("--interpolate-interval-ir", dest = "interpolate_interval_ir", type = int, default = 1) 

parser.add_argument("--range-min-raman", dest= "range_min_raman", type = float, default = 400)
parser.add_argument("--range-max-raman", dest= "range_max_raman", type = float, default = 4000)
parser.add_argument("--interpolate-interval-raman", dest = "interpolate_interval_raman", type = float, default = 1) 

parser.add_argument("--rubberband", dest="rubberband", action="store_true", default=False,)
parser.add_argument("--minmax", dest="minmax", action="store_true", default=False,)
parser.add_argument("--no-interpolate", dest = "no_interpolate", action="store_true", default = False)
parser.add_argument("--nitrogen-calibration", dest="nitrogen_calibration", action="store_true", default=True,)
parser.add_argument("--si-peak", dest= "si_peak", type = float, default = 520)

parser.add_argument("--smooth", dest = "smooth", type = int, default = 0)
parser.add_argument("--baseline", dest = "baseline", type = int, default = 5)


_StoreAction(option_strings=['--baseline'], dest='baseline', nargs=None, const=None, default=5, type=<class 'int'>, choices=None, required=False, help=None, metavar=None)

In [3]:
###Sample Code #1: Specify IR and Raman interpolation intervals along with min & max values
args: Namespace = parser.parse_args([]) ###Do not alter this section.
###PLEASE TYPE IN YOUR ARGUMENTS HERE
args.wn_range_ir_1 = (980, 1800)
args.wn_range_ir_2 = (2720, 2950)
args.range_min_ir = 674.9918 ###enter an integer
args.range_max_ir = 3999.809 ###enter an integer [max value is not included]
args.interpolate_interval_ir = 1.929 ###enter an integer
args.finger_print_exl = True
args.background_rule = 'mean'
args.ths_mean_int_ir = 0.043
args.ths_tot_var_int_ir = 0.64

args.range_min_raman = 355.555542 ###enter an integer or float
args.range_max_raman = 3895.935059 ##enter an integer or float [max value is not included]
args.interpolate_interval_raman = 2.168021680216802 ###enter an integer or float

args.rubberband = 'none' ###True for rubberband, False for iModPoly
args.minmax = 'snv' ###False for SVT normalization, True for minmax
args.no_interpolate = False ###use sample code 2 if using True for this option.
args.nitrogen_calibration = True ##True for calibration, False for no calibration.
args.si_peak = 520 ###enter an integer or float

#directory_name = '/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
directory_name = 'MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
ir_export_filename =  "no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes
file_list_tmp = [] ###option to list out the files you want included. if left as-is, all files in directory will be included.

In [18]:
df_bg_list = []
for f in os.listdir('Background_gloves'):
    df = pd.read_csv('Background_gloves/'+f)
    df = df.set_index('X value [cm⁻¹]').T
    df = df.reset_index(drop=True)
    #df.insert(0, 'id', [f[27:] + '_' + str(i) for i in range(df.shape[0])])
    df_bg_list.append(df)
    
df_bg = pd.concat(df_bg_list, axis=0).reset_index(drop=True)
df_bg['polymer'] = 'bg'
df_bg['map_x'] = 0
df_bg['map_y'] = 0
df_bg

X value [cm⁻¹],948,950,952,954,956,958,960,962,964,966,...,2990,2992,2994,2996,2998,3000,3002,polymer,map_x,map_y
0,-0.101045,0.606542,0.669409,0.198771,-0.287492,0.401664,0.875289,0.221991,-0.369273,-0.055364,...,0.441774,0.896849,1.003423,0.342212,-0.575265,2.839058,-0.298661,bg,0,0
1,0.333104,-0.140061,-0.426264,-0.302323,-0.505759,-0.115593,0.451476,0.504515,0.128778,-0.061872,...,-0.079904,1.375348,1.966410,-0.315713,1.551816,2.345938,3.418875,bg,0,0
2,2.459652,1.825635,1.245425,0.532947,0.452274,0.854054,0.348500,0.229925,0.366944,0.701762,...,1.395473,1.715555,1.725139,1.319857,3.101077,0.714622,2.500347,bg,0,0
3,-0.670037,-1.236778,-0.993990,-0.407710,-0.252830,-0.159352,-0.082116,-0.250556,-0.152791,-0.147242,...,-0.699472,-0.763551,-0.460281,-0.896121,-1.832185,-2.032980,-2.369951,bg,0,0
4,-2.413370,-0.360258,-0.414801,-0.871007,-1.138471,-1.027702,-0.502058,-0.088963,-0.309861,-0.128794,...,0.095599,-0.450613,-1.265870,-1.073055,-1.333631,-2.745036,-1.318634,bg,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,0.416763,-0.176544,-0.001904,0.289580,0.214453,0.229766,0.397104,0.075600,-0.070129,-0.116753,...,0.258797,0.594739,0.702275,-0.725184,-0.549526,0.998947,0.995572,bg,0,0
137,-0.759726,-0.702939,-1.470253,-1.157653,-0.783773,-0.285117,-0.031054,-0.348204,-0.463584,-0.243509,...,0.514787,-0.979082,-0.652291,0.095070,-0.314413,-1.169682,-0.763853,bg,0,0
138,1.433195,1.367441,1.217952,0.445340,0.479496,-0.170026,-0.046395,0.475412,0.422391,0.218491,...,0.016281,0.673870,0.862066,0.497092,-0.657146,-0.429586,0.201705,bg,0,0
139,1.680129,-0.051503,0.392260,0.124603,0.242784,-0.006068,0.198910,0.140559,-0.061062,0.071725,...,0.048430,0.330496,1.080713,1.909293,-0.953686,-0.372122,1.033505,bg,0,0


In [25]:
raman_final, ir_final, label_list, XX_ir, XX_raman, ids, coords, _, rng, ir_bg_label = sp.df_to_interpolated_processed_csv(args, directory_name=None, ir_export_filename=None, raman_export_filename=None, file_list = [[df_bg, df_bg]], transpose = True)


NA Rows Removed but no other issues 0


In [29]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
#preposs_data_ir['id'] = pol_table.iloc[ids,:].reset_index(drop=True)
preposs_data_ir.to_csv('gloves_contamination/data_ir_background_gloves.csv')
preposs_data_ir


,981.7028198242188,983.6317749023438,985.560791015625,987.4898071289062,989.4188232421875,991.3477783203125,993.2767944335938,995.205810546875,997.1348266601562,999.0637817382812,...,2935.77978515625,2937.708740234375,2939.6376953125,2941.56689453125,2943.495849609375,2945.4248046875,2947.353759765625,2949.28271484375,map_x,map_y
0,1.064093,-0.709076,0.148053,-1.478656,-1.931184,-1.414527,-0.648624,0.394164,0.786798,0.429629,...,-1.968301,-1.628936,2.616702,1.884557,-1.023943,0.145100,1.055595,1.230860,0,0
1,0.225281,0.599344,-1.026284,-1.322264,-1.706205,-1.234870,-0.786177,-0.083469,0.360934,-0.075452,...,-1.530050,0.938489,0.075446,2.104574,0.459390,2.036867,2.209021,3.501965,0,0
2,2.140653,1.106025,0.698607,0.334014,1.479439,1.331816,1.045905,0.471054,0.601645,0.421140,...,5.743072,3.300155,2.596484,1.098548,0.038893,1.344325,3.496606,4.225934,0,0
3,-1.811077,-1.197157,0.179185,-0.360507,-0.540565,-1.030370,-0.923846,-1.172802,-1.472101,-1.080226,...,-2.765026,-3.134429,-0.510963,-0.142746,-2.215900,-2.171097,-1.679763,-3.660455,0,0
4,1.005833,0.099599,-0.733488,-1.044153,-0.133176,-0.975554,-2.332681,-2.195930,-0.771425,-0.184024,...,-3.275134,-2.750456,-3.689393,-2.641657,-3.630348,-1.840201,-1.516295,-4.295624,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,-0.224219,0.049475,-0.811554,-0.433064,-1.355675,-1.774116,-0.674873,-0.966245,-0.914912,0.088596,...,0.378480,-2.522405,-0.818144,-4.415222,-4.366511,-2.199884,1.045862,2.266011,0,0
136,1.207168,-0.468039,-2.549086,-1.683398,-1.435072,-1.006440,-0.301754,0.062249,1.783502,1.816475,...,-1.259159,0.558909,2.781947,-0.682063,-1.409779,3.305688,4.938367,0.284720,0,0
137,0.994650,0.461290,1.156144,-0.127088,-1.053293,-0.953797,0.236237,0.548628,1.358646,2.036568,...,0.040222,1.118880,3.262985,4.878158,4.162152,3.761976,1.865993,2.138661,0,0
138,-1.256608,-1.809763,-0.249189,0.387998,0.593958,0.329358,1.097471,2.453407,2.170890,1.201292,...,-2.180602,-1.639424,0.738508,-0.387999,-2.731985,-2.314536,-1.496785,3.112458,0,0
